# ACID Properties

## Overview
**ACID** is the set of four properties that guarantee database transactions are processed reliably. These properties are especially critical in financial systems where partial updates (e.g. debit without credit) would be catastrophic.

| Property | Guarantee | Finance example |
|---|---|---|
| **Atomicity** | All operations in a transaction succeed, or none do | A fund transfer: both debit and credit commit, or neither does |
| **Consistency** | A transaction moves the database from one valid state to another | A balance cannot go below zero (CHECK constraint) |
| **Isolation** | Concurrent transactions do not see each other's uncommitted changes | A balance check during a transfer sees a consistent state |
| **Durability** | Committed transactions survive crashes and power loss | A confirmed deposit is not lost if the server restarts |

**Transaction control statements:**
```sql
BEGIN;               -- start a transaction (SQLite: also BEGIN TRANSACTION)
COMMIT;              -- make all changes permanent
ROLLBACK;            -- undo all changes since BEGIN
SAVEPOINT name;      -- create a partial rollback point (see savepoints notebook)
ROLLBACK TO name;    -- undo back to a savepoint
RELEASE name;        -- release a savepoint
```

**Python sqlite3 context manager (preferred pattern):**
```python
with conn:               # auto-commits on exit, auto-rolls back on exception
    conn.execute("UPDATE ...")
    conn.execute("INSERT ...")
# No need for explicit BEGIN/COMMIT/ROLLBACK when using 'with conn:'
```

---

In [ ]:
import sqlite3

def get_conn():
    conn = sqlite3.connect(":memory:")
    conn.execute("PRAGMA journal_mode=WAL")
    conn.row_factory = sqlite3.Row
    return conn

conn = get_conn()
conn.executescript("""
CREATE TABLE accounts (
    account_id TEXT PRIMARY KEY,
    owner_name TEXT NOT NULL,
    balance    REAL NOT NULL CHECK(balance >= 0),
    version    INTEGER NOT NULL DEFAULT 1
);
CREATE TABLE ledger (
    entry_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    account_id TEXT NOT NULL,
    amount     REAL NOT NULL,
    entry_type TEXT NOT NULL,
    description TEXT
);
INSERT INTO accounts VALUES
    ('ACC001','Aroha Ngata',5000.0,1),
    ('ACC002','Liam Chen',  2500.0,1);
""")
conn.commit()

# ── Helper ────────────────────────────────────────────────────────
def balances(c):
    rows = c.execute("SELECT account_id, owner_name, balance FROM accounts ORDER BY account_id").fetchall()
    for r in rows:
        print(f"  {r['account_id']}  {r['owner_name']:<14s}  ${r['balance']:,.2f}")

print("Initial balances:")
balances(conn)


---
## Atomicity: all-or-nothing

In [ ]:

# ═══ ATOMICITY ════════════════════════════════════════════════════
# All operations in a transaction succeed together, or all are rolled back.
# There is no in-between state visible to other transactions or persistent after failure.

print("\n=== ATOMICITY: transfer $500 from ACC001 to ACC002 ===")
print("Scenario A: both updates succeed — all-or-nothing commit")

def transfer(c, from_id, to_id, amount):
    """Transfer amount between accounts — all-or-nothing."""
    try:
        c.execute("BEGIN")
        c.execute("UPDATE accounts SET balance = balance - ? WHERE account_id = ?", (amount, from_id))
        c.execute("UPDATE accounts SET balance = balance + ? WHERE account_id = ?", (amount, to_id))
        c.execute("INSERT INTO ledger(account_id,amount,entry_type,description) VALUES(?,?,'debit','transfer out')",
                  (from_id, amount))
        c.execute("INSERT INTO ledger(account_id,amount,entry_type,description) VALUES(?,?,'credit','transfer in')",
                  (to_id, amount))
        c.execute("COMMIT")
        print(f"  Transfer of ${amount:,.2f} committed successfully")
    except Exception as e:
        c.execute("ROLLBACK")
        print(f"  Transfer ROLLED BACK: {e}")

transfer(conn, "ACC001", "ACC002", 500.0)
balances(conn)

print("\nScenario B: second update violates CHECK(balance >= 0) — full rollback")
import sqlite3 as _sq3
conn2 = get_conn()
conn2.executescript("""
CREATE TABLE accounts (account_id TEXT PRIMARY KEY, owner_name TEXT NOT NULL,
    balance REAL NOT NULL CHECK(balance >= 0), version INTEGER NOT NULL DEFAULT 1);
INSERT INTO accounts VALUES ('ACC001','Aroha Ngata',200.0,1),('ACC002','Liam Chen',100.0,1);
""")
conn2.commit()
print("  Pre-transfer balances:")
balances(conn2)
transfer(conn2, "ACC001", "ACC002", 500.0)   # tries to take $500 from $200 balance
print("  Post-rollback balances (unchanged):")
balances(conn2)


---
## Consistency and Durability

In [ ]:

# ═══ CONSISTENCY ══════════════════════════════════════════════════
# A transaction brings the database from one valid state to another.
# All constraints (CHECK, NOT NULL, UNIQUE, FK) must hold at commit.

print("\n=== CONSISTENCY: CHECK constraint prevents invalid state ===")
import sqlite3
conn3 = sqlite3.connect(":memory:")
conn3.execute("PRAGMA journal_mode=WAL")
conn3.execute("""CREATE TABLE accounts (
    account_id TEXT PRIMARY KEY,
    balance    REAL NOT NULL CHECK(balance >= 0)
)""")
conn3.execute("INSERT INTO accounts VALUES ('ACC001', 300.0)")
conn3.commit()

print("  Balance before:", conn3.execute("SELECT balance FROM accounts WHERE account_id='ACC001'").fetchone()[0])

try:
    conn3.execute("BEGIN")
    conn3.execute("UPDATE accounts SET balance = -50 WHERE account_id = 'ACC001'")
    conn3.execute("COMMIT")
except sqlite3.IntegrityError as e:
    conn3.execute("ROLLBACK")
    print(f"  Rejected by CHECK constraint: {e}")
    print("  Balance after (unchanged):", conn3.execute("SELECT balance FROM accounts WHERE account_id='ACC001'").fetchone()[0])

print("\n=== DURABILITY: WAL journal mode in SQLite ===")
durability_notes = [
    "Committed transactions survive crashes, power loss, and process kills",
    "SQLite WAL (Write-Ahead Log): PRAGMA journal_mode=WAL — writes go to WAL file first",
    "PostgreSQL: uses WAL (pg_wal/) and fsync to guarantee committed data is on disk",
    "At COMMIT, the database engine ensures data is flushed to durable storage",
    "A crash before COMMIT = transaction never happened (no partial state)",
    "A crash after COMMIT = data is safe (WAL entry is on disk)",
]
print("DURABILITY guarantees:")
for note in durability_notes:
    print(f"  - {note}")


---
## Isolation (preview)

In [ ]:

# ═══ ISOLATION (preview — full coverage in isolation_levels notebook) ══════
# Concurrent transactions do not see each other's uncommitted changes.
# Isolation level controls exactly which phenomena are permitted.

print("=== ISOLATION: concurrent transaction timeline example ===")
print("""
  Time  Transaction A (transfer)          Transaction B (balance check)
  ──────────────────────────────────────────────────────────────────────
  T1    BEGIN
  T2    UPDATE balance -$500 (ACC001)
  T3                                       BEGIN
  T4                                       SELECT balance FROM accounts
                                           (what does B see here?)
  T5    UPDATE balance +$500 (ACC002)
  T6    COMMIT
  T7                                       SELECT balance FROM accounts
  T8                                       COMMIT

  At READ COMMITTED (PostgreSQL default):
    T4: B sees OLD balances (A has not committed yet)
    T7: B sees NEW balances (A committed between T4 and T7) → non-repeatable read

  At REPEATABLE READ (PostgreSQL snapshot):
    T4: B sees OLD balances
    T7: B still sees OLD balances (snapshot taken at T3 is stable)
    → consistent view throughout transaction B

  At READ UNCOMMITTED (not supported in PostgreSQL — conceptual only):
    T4: B could see A's partial changes before A commits → dirty read
""")

print("SQLite isolation: SQLite uses database-level locking.")
print("  In WAL mode: readers do not block writers; writers do not block readers.")
print("  One writer at a time; concurrent readers see the last committed state.")
print("  Effectively SERIALIZABLE for writes, SNAPSHOT for reads.")


---
## Common Pitfalls

**1. Assuming autocommit means transactional safety**
Most database libraries run in autocommit mode by default -- each statement is its own one-statement transaction. A fund transfer written as two separate `UPDATE` statements without an explicit `BEGIN` / `COMMIT` is not atomic: if the process crashes between the two updates, the debit executes but the credit does not. Always wrap multi-statement operations in an explicit transaction.

**2. Catching exceptions without rolling back**
Catching an exception inside a transaction and continuing without `ROLLBACK` leaves the connection in an error state (PostgreSQL) or with uncommitted partial changes (SQLite). The pattern is always: `try: ... COMMIT except: ROLLBACK raise`. In Python, use a context manager (`with conn:`) which auto-commits on success and auto-rolls back on exception.

**3. Confusing consistency in ACID with consistency in CAP theorem**
ACID Consistency means the database moves from one valid state to another (constraints hold). CAP Consistency means all nodes in a distributed system see the same data simultaneously. These are completely different properties. A single PostgreSQL instance provides ACID consistency but is not a distributed system, so CAP does not apply.

**4. Treating durability as guaranteed without proper fsync configuration**
PostgreSQL's `synchronous_commit = off` makes commits faster by not waiting for WAL flush to disk before returning. This risks losing the last few committed transactions on a crash. The default (`synchronous_commit = on`) is durable; only change it for non-critical data.

**5. Very long transactions causing lock accumulation**
An open transaction holds locks (or a snapshot in MVCC systems) for its entire duration. A transaction open for 10 minutes blocks schema changes (ALTER TABLE), autovacuum (PostgreSQL), and in locking systems accumulates row locks that block other writers. Keep transactions as short as possible: do all application logic outside the transaction, open it, write, commit.

**6. Not testing rollback paths**
Most testing verifies the happy path (successful commit). The rollback path -- what happens when a constraint fails, a network error occurs, or a concurrent conflict is detected -- is equally critical in financial systems. Every transaction function should have a test that verifies the database is in the original state after a failure.


---
*sql_methods_library - Samantha McGarrigle*